# Notebook 1 — Project Setup & Exploration

This notebook walks through the foundational setup of the AI Code Review Assistant.

**What you'll do:**
1. Install dependencies
2. Configure your API keys
3. Fetch a real GitHub pull request
4. Explore the raw diff data structure

## Step 1 — Install Dependencies

Run this cell once from the project root.

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-r', '../requirements.txt'], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr[-1000:])

## Step 2 — Configure API Keys

Copy `.env.example` to `.env` and fill in your keys:

```
cp .env.example .env
```

You need:
- **OPENAI_API_KEY** — from https://platform.openai.com/api-keys
- **GITHUB_TOKEN** — from https://github.com/settings/tokens (needs `repo` scope)

Then load them:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

OPENAI_KEY   = os.getenv('OPENAI_API_KEY', '')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')

print('OpenAI key set:' , bool(OPENAI_KEY))
print('GitHub token set:', bool(GITHUB_TOKEN))

## Step 3 — Fetch a Real Pull Request

We'll use a public repo so you can run this without any special access.

In [ ]:
import requests

# Public repo — no token needed for this example
OWNER   = 'pallets'
REPO    = 'flask'
PR_NUM  = 5600  # a real historical Flask PR

headers = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN:
    headers['Authorization'] = f'Bearer {GITHUB_TOKEN}'

pr_url = f'https://api.github.com/repos/{OWNER}/{REPO}/pulls/{PR_NUM}'
pr = requests.get(pr_url, headers=headers).json()

print('PR title  :', pr.get('title', 'N/A'))
print('State     :', pr.get('state'))
print('Files changed:', pr.get('changed_files'))
print('Additions :', pr.get('additions'))
print('Deletions :', pr.get('deletions'))

## Step 4 — Explore the Raw Diff Data

In [ ]:
files_url = f'https://api.github.com/repos/{OWNER}/{REPO}/pulls/{PR_NUM}/files'
files = requests.get(files_url, headers=headers).json()

print(f'Changed files: {len(files)}\n')
for f in files:
    print(f"  {f['status']:10} {f['filename']}  (+{f['additions']} -{f['deletions']})")

In [ ]:
# Inspect the raw patch of the first file
first_file = files[0]
print(f"File: {first_file['filename']}")
print('-' * 60)
print(first_file.get('patch', '(no patch — binary or too large)')[:1500])

## Step 5 — Use Our GitHubClient Class

Now let's use the project module directly.

In [ ]:
from app.github_client import GitHubClient

gh = GitHubClient()
files = gh.get_pull_request_files(OWNER, REPO, PR_NUM)
diff  = gh.build_diff_context(files)

print(f'Diff length: {len(diff)} characters')
print(f'Diff preview:\n{diff[:800]}')

---
**Next:** Open `02_github_webhook_handler.ipynb` to learn how webhooks work.